In [1]:
# === Préparation du dataset pour Power BI ===

import pandas as pd
import os

# Chemins
DATA_CLEAN = '../data/clean/'
POWERBI_DIR = '../powerbi/'

# Création du dossier powerbi s'il n'existe pas
os.makedirs(POWERBI_DIR, exist_ok=True)

# Chargement des datasets nettoyés
df_full = pd.read_csv(DATA_CLEAN + 'imdb_5000_clean.csv')
df_enriched = pd.read_csv(DATA_CLEAN + 'imdb_1000_enriched.csv')

print(f"Dataset complet : {df_full.shape[0]} films")
print(f"Dataset enrichi : {df_enriched.shape[0]} films")

Dataset complet : 4813 films
Dataset enrichi : 1000 films


In [2]:
# === Sélection des colonnes pertinentes pour Power BI ===

colonnes_powerbi = [
    # Identifiants
    'movie_title', 'imdb_id', 'title_year',
    # Caractéristiques
    'duration', 'imdb_score', 'num_voted_users',
    # Équipe
    'director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name',
    # Géographie
    'country', 'language', 'content_rating',
    # Genres splittés
    'genre_1', 'genre_2', 'genre_3',
    # Business
    'budget', 'gross', 'movie_facebook_likes'
]

# On garde uniquement les colonnes qui existent dans le DataFrame
colonnes_dispo = [c for c in colonnes_powerbi if c in df_full.columns]
df_pb = df_full[colonnes_dispo].copy()

# Calcul d'une décennie pour faciliter les filtres
df_pb['decade'] = (df_pb['title_year'] // 10) * 10

# ROI (Return on Investment) si budget et gross dispo
df_pb['profit'] = df_pb['gross'] - df_pb['budget']
df_pb['roi_pct'] = ((df_pb['gross'] - df_pb['budget']) / df_pb['budget'] * 100).round(1)

# Catégorie de note (pour filtrage facile)
def categoriser_note(score):
    if pd.isna(score):
        return 'Non noté'
    if score >= 8:
        return 'Excellent (8+)'
    elif score >= 7:
        return 'Bon (7-8)'
    elif score >= 6:
        return 'Moyen (6-7)'
    else:
        return 'Faible (<6)'

df_pb['categorie_note'] = df_pb['imdb_score'].apply(categoriser_note)

print(f"Colonnes finales : {df_pb.shape[1]}")
print(f"\nColonnes : {df_pb.columns.tolist()}")

Colonnes finales : 23

Colonnes : ['movie_title', 'imdb_id', 'title_year', 'duration', 'imdb_score', 'num_voted_users', 'director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name', 'country', 'language', 'content_rating', 'genre_1', 'genre_2', 'genre_3', 'budget', 'gross', 'movie_facebook_likes', 'decade', 'profit', 'roi_pct', 'categorie_note']


In [3]:
# === Export vers le dossier powerbi/ ===

# Fichier 1 : le dataset principal pour les analyses
df_pb.to_csv(POWERBI_DIR + 'films_dashboard.csv', index=False, encoding='utf-8-sig')
print(f"✅ Exporté : films_dashboard.csv ({len(df_pb)} films)")

# Fichier 2 : table "longue" des genres (pour filtres croisés)
# Chaque ligne = un film + un genre (donc plusieurs lignes par film)
genres_long = []
for _, row in df_pb.iterrows():
    for genre_col in ['genre_1', 'genre_2', 'genre_3']:
        if pd.notna(row[genre_col]):
            genres_long.append({
                'imdb_id': row['imdb_id'],
                'movie_title': row['movie_title'],
                'genre': row[genre_col]
            })

df_genres = pd.DataFrame(genres_long)
df_genres.to_csv(POWERBI_DIR + 'films_genres.csv', index=False, encoding='utf-8-sig')
print(f"✅ Exporté : films_genres.csv ({len(df_genres)} lignes)")

print(f"\n📁 Fichiers prêts dans : {POWERBI_DIR}")

✅ Exporté : films_dashboard.csv (4813 films)
✅ Exporté : films_genres.csv (11920 lignes)

📁 Fichiers prêts dans : ../powerbi/


In [4]:
# === Re-export adapté au format français pour Power BI ===

# Réimport pour repartir propre
df_pb = pd.read_csv(DATA_CLEAN + 'imdb_5000_clean.csv')

# Sélection des colonnes
colonnes_powerbi = [
    'movie_title', 'imdb_id', 'title_year',
    'duration', 'imdb_score', 'num_voted_users',
    'director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name',
    'country', 'language', 'content_rating',
    'genre_1', 'genre_2', 'genre_3',
    'budget', 'gross', 'movie_facebook_likes'
]
colonnes_dispo = [c for c in colonnes_powerbi if c in df_pb.columns]
df_pb = df_pb[colonnes_dispo].copy()

# Recalculs
df_pb['decade'] = (df_pb['title_year'] // 10) * 10
df_pb['profit'] = df_pb['gross'] - df_pb['budget']
df_pb['roi_pct'] = ((df_pb['gross'] - df_pb['budget']) / df_pb['budget'] * 100).round(1)

def categoriser_note(score):
    if pd.isna(score):
        return 'Non noté'
    if score >= 8: return 'Excellent (8+)'
    elif score >= 7: return 'Bon (7-8)'
    elif score >= 6: return 'Moyen (6-7)'
    else: return 'Faible (<6)'

df_pb['categorie_note'] = df_pb['imdb_score'].apply(categoriser_note)

# === EXPORT POWER BI : virgule comme décimal, séparateur point-virgule ===
df_pb.to_csv(
    POWERBI_DIR + 'films_dashboard.csv',
    index=False,
    encoding='utf-8',      # plus de utf-8-sig
    sep=';',               # séparateur point-virgule (standard FR)
    decimal=','            # virgule comme séparateur décimal
)
print(f"✅ films_dashboard.csv ré-exporté au format FR ({len(df_pb)} films)")

# Pareil pour le fichier genres
genres_long = []
for _, row in df_pb.iterrows():
    for genre_col in ['genre_1', 'genre_2', 'genre_3']:
        if pd.notna(row[genre_col]):
            genres_long.append({
                'imdb_id': row['imdb_id'],
                'movie_title': row['movie_title'],
                'genre': row[genre_col]
            })

df_genres = pd.DataFrame(genres_long)
df_genres.to_csv(
    POWERBI_DIR + 'films_genres.csv',
    index=False,
    encoding='utf-8',
    sep=';',
    decimal=','
)
print(f"✅ films_genres.csv ré-exporté au format FR ({len(df_genres)} lignes)")

✅ films_dashboard.csv ré-exporté au format FR (4813 films)
✅ films_genres.csv ré-exporté au format FR (11920 lignes)
